In [ ]:
import sys
!{sys.executable} -m pip install --upgrade protobuf

  Using cached pandas-3.0.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached tensorflow-2.21.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (4.4 kB)
  Using cached tensorflowjs-4.22.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached numpy-2.4.4-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.34.1-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
  Using cached r

In [2]:

import pandas as pd
import tensorflow as tf
import tensorflowjs as tfjs
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import json
import os

# 1. Tải và chuẩn bị dữ liệu (Giống hệt cách ông làm)
df = pd.read_csv("./train.csv")

# Chuyển đổi nhãn (Label)
# C = 0 (Sai form hoặc Gập), L = 1 (Đúng form hoặc Duỗi) - Tùy logic của ông
df.loc[df["label"] == "C", "label"] = 0
df.loc[df["label"] == "L", "label"] = 1

# Tách Features (36 cột tọa độ) và Label
X = df.drop("label", axis=1).values
y = df["label"].astype('int').values

# 2. Xây dựng Scaler BẰNG TAY để mang qua JS
sc = StandardScaler()
X_scaled = sc.fit_transform(X)

# Lưu thông số Scaler ra file JSON
os.makedirs("tfjs_bicep_posture", exist_ok=True)
scaler_data = {
    "mean": sc.mean_.tolist(),
    "scale": sc.scale_.tolist()
}
with open("tfjs_bicep_posture/scaler.json", "w") as f:
    json.dump(scaler_data, f)
print("✅ Đã lưu Scaler!")

# 3. Chia tập Train/Test (Đảm bảo model không học vẹt)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 4. Tạo Mô hình Phân loại (Thay thế KNN)
# Đây là linh hồn của dự án Hackathon: Neural Network cho Form Detection
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(36,)), # Đầu vào chính xác 36 cột
    tf.keras.layers.Dropout(0.2), # Chống overfit
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid') # Phân loại 2 lớp (C và L)
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("🔥 Bắt đầu huấn luyện AI...")
model.fit(X_train, y_train, epochs=100, batch_size=16, validation_data=(X_test, y_test))

# 5. Xuất Model ra định dạng cho React Native
tfjs.converters.save_keras_model(model, "tfjs_bicep_posture")
print("🎉 THÀNH CÔNG! Hãy lấy thư mục tfjs_bicep_posture đưa vào React Native.")

VersionError: Detected incompatible Protobuf Gencode/Runtime versions when loading yggdrasil_decision_forests/dataset/data_spec.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.